In [1]:
!pip install selenium
!apt-get update
!apt install chromium-chromedriver

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 476.0/476.0 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 2.8 MB/s eta 0:00:00
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Ign:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy Release [5,713 B]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy Release.gpg [793 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [57.5 kB]
Hit:11 https://ppa.launchpadcont

In [2]:
import csv
import os
import time
from datetime import date, datetime
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from concurrent.futures import ThreadPoolExecutor
import shutil
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive')

# Inicialize o WebDriver
def iniciar_driver():
    opções = webdriver.ChromeOptions()
    opções.add_argument('--headless')
    opções.add_argument('--no-sandbox')
    opções.add_argument('--disable-dev-shm-usage')
    driver = webdriver.Chrome(options=opções)
    driver.implicitly_wait(5)
    return driver

# Função para coletar produtos de uma categoria
def coletar_produtos(categoria, url):
    driver = iniciar_driver()
    produtos = []
    produtos_carregados = set()  # Usaremos um set para rastrear produtos únicos
    pagina_atual = 1

    print(f"\033[94mIniciando a coleta de produtos da categoria: {categoria}\033[0m")

    while True:
        try:
            # Navegue até a página atual
            url_com_pagina = f"{url.split('?')[0]}?page={pagina_atual}"
            driver.get(url_com_pagina)

            # Aguarde até que os produtos sejam carregados
            produtos_atual = driver.find_elements(By.CSS_SELECTOR, '.vtex-search-result-3-x-galleryItem')

            # Adicione os produtos à lista
            novos_produtos = False
            for produto in produtos_atual:
                try:
                    # Coletar informações do produto
                    nome_empresa = "Joli"
                    data_coleta = date.today().strftime("%Y-%m-%d")
                    codigo_produto = produto.find_element(By.CSS_SELECTOR, '.vtex-product-summary-2-x-brandName').text.strip()
                    imagem_url = produto.find_element(By.CSS_SELECTOR, '.vtex-product-summary-2-x-image').get_attribute('src')
                    nome_produto = produto.find_element(By.CSS_SELECTOR, '.vtex-product-summary-2-x-productNameContainer').text.strip()
                    moeda = produto.find_element(By.CSS_SELECTOR, '.joli-joli-theme-0-x-price').text.strip()
                    url_produto = produto.find_element(By.CSS_SELECTOR, 'a').get_attribute('href')

                    # Montar a linha de dados do produto
                    linha_produto = [nome_empresa, data_coleta, codigo_produto, imagem_url, nome_produto, moeda, url_produto]

                    # Adicionar à lista de produtos se ainda não estiver na lista
                    if nome_produto not in produtos_carregados:
                        produtos.append(linha_produto)
                        produtos_carregados.add(nome_produto)
                        novos_produtos = True

                except NoSuchElementException:
                    continue

            print(f"\033[93mTotal de produtos coletados de {categoria} até agora: {len(produtos)}\033[0m")

            if not novos_produtos:
                print(f"\033[93mNão há mais novos produtos para carregar de {categoria}.\033[0m")
                break

            # Avance para a próxima página
            pagina_atual += 1

        except TimeoutException:
            print(f"\033[91mNão foi possível encontrar mais produtos em {categoria}.\033[0m")
            break

    driver.quit()

    # Salvar os produtos em um arquivo CSV com nome da categoria
    nome_arquivo = os.path.join("output", f'produtos_{categoria}.csv')
    with open(nome_arquivo, 'w', newline='', encoding='utf-8') as arquivo_csv:
        escritor_csv = csv.writer(arquivo_csv)
        escritor_csv.writerow(['Nome empresa', 'Data de coleta', 'Código do produto', 'Imagem', 'Nome do produto', 'Moeda', 'URL do produto'])

        for produto in produtos:
            escritor_csv.writerow(produto)

    print(f"\033[92mOs produtos da categoria {categoria} foram salvos em: {nome_arquivo}\033[0m")


# Criar pasta de saída se não existir
output_folder = "output"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"\033[92mA pasta {output_folder} foi criada.\033[0m")

# Dicionário de URLs por categoria
urls = {
    "Cozinhas e areas de servico": "https://www.joli.com.br/banheiros-cozinhas-e-areas-de-servico?page=1",
    "Bomboniere" : "https://www.joli.com.br/bomboniere?page=1",
    "Chuveiros e aquecedores" : "https://www.joli.com.br/chuveiros-e-aquecedores?page=1",
    "climatizacao e ventilacao" : "https://www.joli.com.br/climatizacao-e-ventilacao?page=1",
    "decoracao e utilidades domesticas" : "https://www.joli.com.br/decoracao-e-utilidades-domesticas?page=1",
    "Eletrica e iluminacao" : "https://www.joli.com.br/eletrica-e-iluminacao?page=1",
    "Ferramentas" : "https://www.joli.com.br/ferramentas-e-ferragens?page=1",
    "Jardim" : "https://www.joli.com.br/lazer-e-jardim?page=1",
    "Material de construcao" : "https://www.joli.com.br/material-de-construcao?page=1",
    "Material hidraulico" : "https://www.joli.com.br/material-hidraulico?page=1",
    "Pintura e acessorios" : "https://www.joli.com.br/pintura-e-acessorios?page=1",
    "Pisos revistimentos e porcelanatos" : "https://www.joli.com.br/pisos-revestimentos-e-porcelanatos?page=1",
    "Portas e janelas" : "https://www.joli.com.br/portas-e-janelas?page=1",
    "Seguranca e comunicacao" : "https://www.joli.com.br/seguranca-e-comunicacao?page=1"
}

# Usar múltiplas threads para acelerar a coleta de dados
with ThreadPoolExecutor(max_workers=5) as executor:
    for categoria, url in urls.items():
        executor.submit(coletar_produtos, categoria, url)

# Mover arquivos CSV para o Google Drive
base_destination_dir = '/content/drive/MyDrive/WebScraping/Joli/WebScraping_coleta_Joli'
today = datetime.today().strftime('%Y-%m-%d')
destination_dir = os.path.join(base_destination_dir, today)

if not os.path.exists(destination_dir):
    os.makedirs(destination_dir, exist_ok=True)

source_dir = output_folder

for file_name in os.listdir(source_dir):
    full_file_name = os.path.join(source_dir, file_name)
    if os.path.isfile(full_file_name):
        shutil.copy(full_file_name, destination_dir)

print(f"\033[92mArquivos CSV movidos para: {destination_dir}\033[0m")

Mounted at /content/drive
A pasta output foi criada.
Iniciando a coleta de produtos da categoria: decoracao e utilidades domesticas
Iniciando a coleta de produtos da categoria: climatizacao e ventilacao
Iniciando a coleta de produtos da categoria: Cozinhas e areas de servico
Iniciando a coleta de produtos da categoria: Bomboniere
Iniciando a coleta de produtos da categoria: Chuveiros e aquecedores
Total de produtos coletados de Bomboniere até agora: 6
Total de produtos coletados de climatizacao e ventilacao até agora: 6
Total de produtos coletados de decoracao e utilidades domesticas até agora: 6
Total de produtos coletados de Cozinhas e areas de servico até agora: 6
Total de produtos coletados de Chuveiros e aquecedores até agora: 6


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "<ipython-input-2-18b0316f2e58>", line 122, in <cell line: 122>
    with ThreadPoolExecutor(max_workers=5) as executor:
  File "/usr/lib/python3.10/concurrent/futures/_base.py", line 649, in __exit__
    self.shutdown(wait=True)
  File "/usr/lib/python3.10/concurrent/futures/thread.py", line 235, in shutdown
    t.join()
  File "/usr/lib/python3.10/threading.py", line 1096, in join
    self._wait_for_tstate_lock()
  File "/usr/lib/python3.10/threading.py", line 1116, in _wait_for_tstate_lock
    if lock.acquire(block, timeout):
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/IPython/core/interactiveshell.py", line 2099, in showtraceback
    stb = value._render_traceb

TypeError: object of type 'NoneType' has no len()